<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 3 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">模型语义与分区分桶</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">观察相同键在三种模型下的结果，理解分区与分桶如何缩小查询范围。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">目标 Doris 4.1.3 · 订单数据 · 独立实验库</span>
</div>

完成后，你将比较三种表模型对重复键的处理，并通过查询计划观察日期分区与分桶裁剪。请按顺序运行。

[讲义](course3_models_partitioning_and_bucketing.md) · [课程入口](../README.md)


## 实验范围

仅重建 orders_duplicate、orders_unique、orders_aggregate、orders_partitioned。重复 Key 的语义和数据分布是两个独立问题。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()




## 1. 相同键，不同模型

从 WWI 样本取订单 1（2300.00）和订单 2（405.00），在实验表中模拟将订单 1 修正为 2250.00。按顺序完成三次提交后，观察：

- Duplicate Key 保留三条输入。
- Unique Key 保留两笔订单，订单 1 金额为 2250.00。
- Aggregate Key 按键累加，订单 1 得到 4550.00。

最后一种结果适合累加指标；要表达订单修正后的当前金额，应使用按键更新的语义。


In [ ]:
definitions = {
    "orders_duplicate": 'CREATE TABLE orders_duplicate (id BIGINT, amount DECIMAL(12,2)) DUPLICATE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1")',
    "orders_unique": 'CREATE TABLE orders_unique (id BIGINT, amount DECIMAL(12,2)) UNIQUE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")',
    "orders_aggregate": 'CREATE TABLE orders_aggregate (id BIGINT, amount DECIMAL(12,2) SUM) AGGREGATE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1")',
}
for table, ddl in definitions.items():
    lab.execute(f"DROP TABLE IF EXISTS {table}")
    show_sql("建表 SQL", ddl)
    lab.execute(ddl)
    for row in [(1, "2300.00"), (1, "2250.00"), (2, "405.00")]:
        lab.insert(table, ["id", "amount"], [row])
expect(lab.query("SELECT id, amount FROM orders_duplicate ORDER BY id, amount"), [(1,"2250.00"),(1,"2300.00"),(2,"405.00")])
expect(lab.query("SELECT id, amount FROM orders_unique ORDER BY id"), [(1,"2250.00"),(2,"405.00")])
expect(lab.query("SELECT id, amount FROM orders_aggregate ORDER BY id"), [(1,"4550.00"),(2,"405.00")])


## 2. 分区与分桶各管什么

分区表使用原始十笔订单：每天一个分区，每个分区按订单号 Hash 分成四个桶。依次比较全表查询、只过滤日期、同时过滤日期和订单号的计划，查看所选分区与 Tablet 范围如何缩小。

这张表保留历史明细，日期和订单号用于排序；订单当前表的业务唯一键需要单独设计。


In [ ]:
from dw_course.wwi import sample
lab.execute("DROP TABLE IF EXISTS orders_partitioned")
lab.execute("""
CREATE TABLE orders_partitioned (
    order_date DATE, order_id BIGINT, amount DECIMAL(18,2)
) DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p_day1 VALUES [('2013-01-01'), ('2013-01-02')),
    PARTITION p_day2 VALUES [('2013-01-02'), ('2013-01-03'))
)
DISTRIBUTED BY HASH(order_id) BUCKETS 4
PROPERTIES("replication_num"="1")
""")
lab.insert("orders_partitioned", ["order_date", "order_id", "amount"],
           [(r["order_date"], r["order_id"], r["order_amount"]) for r in sample()["orders"]])
for query in [
    "SELECT * FROM orders_partitioned",
    "SELECT * FROM orders_partitioned WHERE order_date = '2013-01-01'",
    "SELECT * FROM orders_partitioned WHERE order_date = '2013-01-01' AND order_id = 1",
]:
    lab.sql("EXPLAIN " + query)
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM orders_partitioned WHERE order_date = '2013-01-01'"),
       [(5, "3944.20")])
lab.close()


## 完成标准

核对三种模型的订单数及金额，在 EXPLAIN 的扫描节点中找到所选分区和 Tablet。关注扫描范围的变化，字段名称和计划排版可能随版本变化。


## 自己动手：哪个结果可以给业务？

Aggregate Key 的 4550.00 来自 2300.00 + 2250.00。若报表想查订单 1 的当前金额，为什么这个结果不合适？再查询分区表第一天的金额，应为原始样本的 3944.20。
